In [1]:
%pip install google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

from dotenv import find_dotenv, load_dotenv
from google import genai
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd

load_dotenv(find_dotenv(usecwd=True))
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("Set GOOGLE_API_KEY in a .env file or the environment.")

gemini_client = genai.Client(api_key=api_key)

In [3]:
df_items = pd.read_json("../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [4]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Electronics,2 Pack-iPhone Earbuds Wired Lightning Headphon...,3.4,598,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],WASABI MANGO,"[Electronics, Headphones, Earbuds & Accessorie...",{'Product Dimensions': '23.62 x 19.69 x 27.56 ...,B0B1ZVC7GJ,NaN,NaN,NaN
1,Computers,"Mini PC 16GB DDR4 256GB M.2 SSD,Quad-Core 2.7G...",4.3,450,[【Meet to Sufficient Memory Storage】This Mini ...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],OUVISLITE,"[Electronics, Computers & Accessories, Compute...","{'Screen Resolution': '3840 x 2160', 'Max Scre...",B0B1HNV2V9,NaN,NaN,NaN
2,Computers,Samsers Foldable Bluetooth Keyboard with Touch...,4.4,308,[【Full-size Folding Wireless Keyboard】 Samsers...,[],48.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Love this keyboard!', 'url': 'http...",Samsers,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '13.5 x 4.5 x 0.4 inche...,B0C2Q8BDTX,NaN,NaN,NaN
3,Computers,"Rolling Laptop Bag Women with Wheels, Rolling ...",4.5,152,[MOBILE OFFICE: EMPSIGN rolling bag with lapto...,[rolling laptop bag],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Elegant Laptop Tote Bag for Women ...,Ytonet,"[Electronics, Computers & Accessories, Laptop ...","{'Brand': 'Ytonet', 'Item model number': '1332...",B092Z9CTNK,NaN,NaN,NaN
4,All Electronics,"Wireless Mouse, 2.4G Silent Mouse with USB Rec...",4.6,677,[Plug & Play Super Easy to Use- Just plug and ...,[],10.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],MagoFeliz,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '4.69 x 2.6 x 0.01 inch...,B0C77L1G7V,NaN,NaN,NaN


In [5]:
from typing import Any, Hashable

list[tuple[Hashable, Any]](df_items["features"].items())[0:]

[(0, []),
 (1,
  ['【Meet to Sufficient Memory Storage】This Mini Computer equiped with 16GB DDR4 and 256GB SSD. While ensuring cost-effectiveness, the larger capacity brings you a better multitasking experience. Newbie friendly Storage upgrade:1)2.5 inch SATA SSD/HDD up to 2TB(not include). 2)replace the M.2 SATA HDD. Expand your applications, files, video or audio freely.',
   '【J4125 Mini PC with Windows 11 Pro】Celeron Gemini Lake J4125 Quad-Core processor with base frequency of 2.0 GHz and up to 2.7 GHz in burst. Let the minis to run multiple applications, Web pages, full HD video or light game smoothly. Windows 11 preinstalled and automatically activated after connecting to the Internet, plug and play!',
   '【Enjoy Immersive Visual Experience】The desktop computer has 2 HDMI ports and 1 VGA port for triple screen output, supporting three-screen simultaneous display or different display. The combination with HD Graphics 600 (up to 750MHz) makes work more efficient, 4K@60Hz HD brings y

In [6]:
list[tuple[Hashable, Any]](df_items["images"].items())[0]

(0,
 [{'thumb': 'https://m.media-amazon.com/images/I/31eziY1O3EL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/31eziY1O3EL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/51tDCwwyEgL._AC_SL1417_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51J0Iti28iL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51J0Iti28iL._AC_.jpg',
   'variant': 'PT01',
   'hi_res': 'https://m.media-amazon.com/images/I/71voGQrX7nL._AC_SL1276_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/419vSzdw3pL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/419vSzdw3pL._AC_.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/61Iub5dt2SL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/4111pWhpUIL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/4111pWhpUIL._AC_.jpg',
   'variant': 'PT03',
   'hi_res': 'https://m.media-amazon.com/images/I/612nAIai-WL._AC_SL1

In [7]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [8]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [9]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [10]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,image
0,All Electronics,2 Pack-iPhone Earbuds Wired Lightning Headphon...,3.4,598,[],2 Pack-iPhone Earbuds Wired Lightning Headphon...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],WASABI MANGO,"[Electronics, Headphones, Earbuds & Accessorie...",{'Product Dimensions': '23.62 x 19.69 x 27.56 ...,B0B1ZVC7GJ,NaN,NaN,NaN,https://m.media-amazon.com/images/I/31eziY1O3E...
1,Computers,"Mini PC 16GB DDR4 256GB M.2 SSD,Quad-Core 2.7G...",4.3,450,[【Meet to Sufficient Memory Storage】This Mini ...,"Mini PC 16GB DDR4 256GB M.2 SSD,Quad-Core 2.7G...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],OUVISLITE,"[Electronics, Computers & Accessories, Compute...","{'Screen Resolution': '3840 x 2160', 'Max Scre...",B0B1HNV2V9,NaN,NaN,NaN,https://m.media-amazon.com/images/I/31d48aRFMM...
2,Computers,Samsers Foldable Bluetooth Keyboard with Touch...,4.4,308,[【Full-size Folding Wireless Keyboard】 Samsers...,Samsers Foldable Bluetooth Keyboard with Touch...,48.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Love this keyboard!', 'url': 'http...",Samsers,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '13.5 x 4.5 x 0.4 inche...,B0C2Q8BDTX,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51bW+ihg8x...
3,Computers,"Rolling Laptop Bag Women with Wheels, Rolling ...",4.5,152,[MOBILE OFFICE: EMPSIGN rolling bag with lapto...,"Rolling Laptop Bag Women with Wheels, Rolling ...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Elegant Laptop Tote Bag for Women ...,Ytonet,"[Electronics, Computers & Accessories, Laptop ...","{'Brand': 'Ytonet', 'Item model number': '1332...",B092Z9CTNK,NaN,NaN,NaN,https://m.media-amazon.com/images/I/01RmK+J4pJ...
4,All Electronics,"Wireless Mouse, 2.4G Silent Mouse with USB Rec...",4.6,677,[Plug & Play Super Easy to Use- Just plug and ...,"Wireless Mouse, 2.4G Silent Mouse with USB Rec...",10.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],MagoFeliz,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '4.69 x 2.6 x 0.01 inch...,B0C77L1G7V,NaN,NaN,NaN,https://m.media-amazon.com/images/I/31cV2lTLOG...


In [11]:
df_sample = df_items.sample(50, random_state=42)

In [12]:
len(df_sample)

50

In [13]:
data_to_embed = df_sample[["description", "image", "rating_number", "price", "average_rating", "parent_asin"]].to_dict(orient="records")

In [14]:
data_to_embed

[{'description': 'YKZ 3 in 1 Magnetic C to C Cable 4 Pack, Magnet Type C to Type C Charging Cable, [3.3FT+6.0FT] PD 60W Fast Charger Nylon Braided Cord for Type C/MicroUSB and i-Product 【3 in 1 USB C to USB C cable】- This cable has 3 different magnet connectors, suitable for Type-c, Micro-USB devices and i-Product. (NOTE: This is the USB-C output cable, you may need a USB C wall charger to charge your device.) 【PD 60W Fast Charge】- YKZ type C to type C Cable supports Power Delivery fast charge up to 20V/3A (max), 15V/3A, 12V/3A, 9V/3A, 5V/3A at 60W with USB-C PD adapters such as PD 18W/30W/45W/60W. (Note: Charging speeds also depends on your device and charger. ) 【Magnetic Charging Cable】- Magnetic charging cable has a strong magnet that can attract the phone without falling off. You can easily connect with one hand while driving or doing other things. Besides, Magnetic tips can be used as a dust plug that reducing abrasion during inserting or plugging out the cable, greatly prolong th

Create a Gemini embedding

In [15]:
import os
from dotenv import find_dotenv, load_dotenv
from google import genai

load_dotenv(find_dotenv(usecwd=True))

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("Set GOOGLE_API_KEY in a .env file or the environment.")

gemini_client = genai.Client(api_key=api_key)

In [16]:
response = gemini_client.models.embed_content(
    model="gemini-embedding-001",
    contents="Random text",
)
embedding = response.embeddings[0].values
len(embedding), embedding[:5]

(3072, [-0.014552347, 0.031574354, 0.012304088, -0.048244637, 0.0036319043])

In [17]:
embedding = response.embeddings[0].values
len(embedding)

3072

In [18]:
def get_embedding(text: str, model: str = "gemini-embedding-001") -> list[float]:
    response = gemini_client.models.embed_content(
        model=model,
        contents=text,
    )
    return response.embeddings[0].values

In [19]:
get_embedding("HELlo")

[-0.03937038,
 0.022121867,
 0.014735156,
 -0.072452515,
 -0.013474568,
 -0.014287574,
 -0.007858721,
 0.013496504,
 -0.0037166376,
 -0.003122414,
 -0.004082619,
 -0.011833767,
 0.00047123732,
 0.01938954,
 0.109334,
 0.0014172449,
 0.010168162,
 0.012269794,
 -0.017635757,
 -0.026045257,
 -0.004717633,
 -0.004786045,
 0.0009797459,
 0.0019292262,
 -0.0131798275,
 0.0031843006,
 0.019197434,
 0.0007098973,
 0.02581244,
 0.0021720142,
 0.02185504,
 -0.013525502,
 0.0007759163,
 0.0015975212,
 0.0081598,
 0.003161199,
 0.019949993,
 -0.0023752437,
 -0.013870821,
 0.022827838,
 -0.002327836,
 0.007240768,
 0.017104527,
 -0.016749613,
 -0.013075054,
 -0.0044487487,
 -0.004539151,
 -0.023608582,
 -0.0068665976,
 0.04340169,
 -0.0021608802,
 -9.624353e-05,
 -0.008950546,
 -0.13380873,
 -0.018240664,
 0.0046943706,
 -0.006606803,
 0.010006489,
 -0.0030065533,
 -0.017445333,
 0.016505564,
 0.0082285395,
 0.0025865068,
 -0.036564298,
 0.027723363,
 0.016236449,
 -0.015749255,
 0.011425523,
 -0.

In [20]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [30]:
collection_name = "Amazon-items-collection-00"

if qdrant_client.collection_exists(collection_name):
    qdrant_client.delete_collection(collection_name)

qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
)

True

In [31]:
pointstruct = PointStruct(
    id=0,
    vector=get_embedding("Test text"),
    payload={
        "text": "Test text",
        "model": "text-embedding-3-small",
    },
)

In [32]:
pointstruct

PointStruct(id=0, vector=[-0.0131258825, 0.014974858, 0.014586804, -0.05772458, 0.00080141576, 0.006194405, 0.02275859, -0.010324336, -0.0042547868, 0.0018398004, -0.012825156, -0.030300578, 0.01593786, 0.020253858, 0.11358426, 0.0046583475, -0.017471893, 0.02234187, -0.015648505, -0.014378464, 0.0120845875, -0.018523809, 0.0045633167, -0.0025803102, -0.00037962027, 0.010354163, 0.026146907, 0.016915768, 0.034327187, -0.010151756, 0.005513112, 0.017949905, 0.020580065, -0.0027162207, 0.007837247, 0.002970026, -0.012072991, 0.0235855, -0.009860177, 0.0073412964, -0.0024134682, -0.012781421, -0.0036442238, -0.0055578332, 0.02275483, -0.0075101107, 0.029694026, -0.026103249, 0.0039535514, 0.01774619, 0.0065223454, -0.011149641, -0.02212571, -0.15625586, 0.0037094734, -0.035037607, -0.020441994, 0.020114006, -0.0042393473, 0.019911582, -0.015497046, 0.009644813, -0.010357382, -0.02036744, -0.014241611, -0.012409947, -0.01807379, 0.029578017, -0.008487547, 0.009201351, -0.0012298817, -0.006

In [33]:
pointstructs = []
for i, data in enumerate[dict](data_to_embed):
    embediing = get_embedding(data["description"])
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embediing,
            payload=data,
        )
    )

In [34]:
pointstructs


[PointStruct(id=0, vector=[0.00046646033, -0.041947875, 0.024622949, -0.041609425, 0.028612616, 0.018302882, 0.020005964, -0.0061465497, -0.001811255, 0.012784808, 0.0131245265, -0.031175544, 0.02043033, -0.020321311, 0.1045425, 0.009155745, 0.0028868713, -0.033774152, -0.006822222, 0.006882424, -0.03114653, 0.010464398, 0.001883688, 0.0056899893, -0.0014231623, -0.0017375277, 0.0036751812, 0.0033075835, 0.035230212, 0.0052612806, -0.039177716, 0.041192725, 0.009940649, 0.0109757045, 0.029805925, -0.00029353687, 0.009646634, -0.011479955, -0.0012151056, 0.00846899, 0.008363996, 0.032164194, 0.02259798, 0.014176407, -0.027313393, -0.024754073, -0.00032623688, 0.033928815, 0.007884388, 0.002716877, -0.022488784, 0.006250891, 0.022240344, -0.18290752, 0.002498773, -0.021384839, -0.007167023, -0.0096356245, -0.02751605, -0.0067681023, -0.026164625, 0.05017759, -0.01620066, -0.019171942, -0.026374323, -0.009801013, 0.0052320845, 0.0057560294, 0.005550437, 0.012435091, 0.027679045, 0.0122874

In [35]:
len(pointstructs)

50

In [36]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-00",
    wait=True,
    points=pointstructs,
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [37]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k,
    )
    return results

In [38]:
retrieve_data("What kind of charging ports do you offer? ", k=10).points

[ScoredPoint(id=39, version=1, score=0.6031601, payload={'description': '45W USB C Charger Compatible with Hp/Dell/Lenovo/Acer chromebook/Asus/Samsung/Sony Charger Fast Charging Type C AC Adapter Power Cord Supply 【Details】Input: AC 100-240V~1.5A 56-60Hz Output: 5V-3A / 9V-3A / 12V-3A / 15V-3A / 18V-2.5A / 20V-2.25A Power: 10W/36W/45W;Plug Size: USB-C tip Plug Type: USB-C / Type C Wide Compatibility: Standard USB Type-C Plug, Widely Compatible with: HP, Dell, Lenovo, Acer, Asus, Samsung, Google, others and more 【Easy to Use and Carry with You 】45W Type-C AC Charger is ready to go anywhere you do. It’s the perfect partner whenever, and wherever you need to charge. CE-/FCC-/RoHS-Certified for safety. 24 x 7 email support,12 Months warranty, 30 Days Money Back Guarantee! Buy with confidence!Please contact us with any questions or concerns', 'image': 'https://m.media-amazon.com/images/I/41xG67xMIYL._AC_.jpg', 'rating_number': 174, 'price': 11.99, 'average_rating': 4.3, 'parent_asin': 'B0B2